# AI512 Project: UltraLight VM-UNet on TN3K Thyroid Ultrasound
## Lesion-Scale-Aware Boundary Preservation under Ultrasound Speckle Degradation

### End-to-End Orchestrator for Experiments E1–E4
- **E1-R***: Baseline UltraLight VM-UNet reproduction under standardized shared recipe (BCE+Dice, 200 epochs, cosine LR with warm-up, EMA, gradient clipping, deep supervision)
- **E2**: Shared recipe + Lesion-Scale-Aware Boundary Supervision (`boundary_mode: from_mask`, normalized scale weights)
- **E3**: Shared recipe + Corruption-Aware Consistency Regularization (SoftDice + MSE on soft probability maps under random speckle/contrast perturbations)
- **E4**: Full Combined Framework (E2 Boundary + E3 Consistency)

> **Robust Checkpointing & Code Versioning**: Each experiment tracks its git commit hash (`code_version`) and resolved configuration. Checkpoints are saved under `{exp_name}/seed{seed}/`. Existing completed runs matching the code version are skipped safely; if code version differs, retraining is triggered to prevent stale results.

### 1. GPU & Hardware Verification

In [ ]:
!nvidia-smi

### 2. Environment Setup & Dependency Installation

In [ ]:
!pip install -q pyyaml scipy scikit-image timm thop pandas matplotlib opencv-python pytest
print("✅ Dependencies successfully installed!")

### 3. Sync Repository & Run Unit Test Verification Suite

In [ ]:
import os
REPO_URL = "https://github.com/YeswanthTejaJakka/ML_UVnet_Project.git"

if not os.path.exists('/content/ML_UVnet'):
    !git clone $REPO_URL /content/ML_UVnet
else:
    %cd /content/ML_UVnet
    !git pull

%cd /content/ML_UVnet

# Run P1 & P6 correctness suites
!python tests/test_pscan.py
!python tests/test_metrics.py
print("\n✅ All numerical, architecture, and metric unit tests passed!")

### 4. Mount Google Drive & Prepare TN3K Dataset

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')

drive_zip = '/content/drive/MyDrive/TN3K.zip'
drive_folder = '/content/drive/MyDrive/TN3K_extracted/Thyroid Dataset/tn3k'
local_dir = '/content/tn3k'

if not os.path.exists(local_dir):
    if os.path.exists(drive_zip):
        print("Unpacking TN3K.zip from Drive to local NVMe SSD...")
        !unzip -q "$drive_zip" -d /content/
        if not os.path.exists(local_dir) and os.path.exists('/content/Thyroid Dataset/tn3k'):
            !mv "/content/Thyroid Dataset/tn3k" /content/tn3k
        elif not os.path.exists(local_dir) and os.path.exists('/content/TN3K'):
            !mv /content/TN3K /content/tn3k
    elif os.path.exists(drive_folder):
        print("Copying extracted dataset from Drive...")
        !cp -r "$drive_folder" /content/tn3k

print("\n--- Dataset Path Verification ---")
print("TrainVal Images:", len(os.listdir('/content/tn3k/trainval-image')) if os.path.exists('/content/tn3k/trainval-image') else 'MISSING')
print("Test Images:    ", len(os.listdir('/content/tn3k/test-image')) if os.path.exists('/content/tn3k/test-image') else 'MISSING')
print("Fold 0 JSON:    ", os.path.exists('/content/tn3k/tn3k-trainval-fold0.json'))

### 5. P0: Dataset Integrity & Leakage Audit
Audits file counts, fold-JSON match rates, MD5 hash disjointness vs test set, and lesion scale distributions.

In [ ]:
%cd /content/ML_UVnet
!python tools/audit_dataset.py --data_root /content/tn3k --fold 0 --out_dir audit

with open('audit/audit_report.md', 'r') as f:
    print(f.read())

### 6. Quick Smoke Test (2 Epochs across E1–E4)
Verifies the full end-to-end training and deliverable pipeline in ~2 minutes before starting full runs.

In [ ]:
%cd /content/ML_UVnet
# Run smoke test
!python run_all.py --stage smoke --data_root /content/tn3k

### 7. Stage A: Recipe Ladder (Validation Only, Seed 42, 100 Epochs)
Measures incremental impact of each design component (R0 -> R1 -> R2 -> R3 -> R4).

In [ ]:
%cd /content/ML_UVnet
!python run_all.py --stage A --data_root /content/tn3k --checkpoints_root /content/drive/MyDrive/ML_UVnet_checkpoints

if os.path.exists('reports/recipe_ladder.csv'):
    import pandas as pd
    display(pd.read_csv('reports/recipe_ladder.csv'))

### 8. Stage B: Loss Hyperparameter Tuning (Validation Only, Seed 42)
Tuning grids: $\lambda_{boundary} \in \{0.1, 0.3, 0.5\}$ and $\lambda_{consistency} \in \{0.5, 1.0, 2.0\}$.

In [ ]:
%cd /content/ML_UVnet
!python run_all.py --stage B --data_root /content/tn3k --checkpoints_root /content/drive/MyDrive/ML_UVnet_checkpoints

if os.path.exists('reports/stage_b_tuning.csv'):
    import pandas as pd
    display(pd.read_csv('reports/stage_b_tuning.csv'))

### 9. Stage C: Full Benchmark Experiments (E1-R*, E2, E3, E4 across Seeds 42, 43, 44)
Runs full training using the canonical YAML configs (200 epochs, bs 8, warmup + cosine schedule, EMA).

In [ ]:
%cd /content/ML_UVnet
CKPT_ROOT = "/content/drive/MyDrive/ML_UVnet_checkpoints"

# Run final experiments across seeds 42, 43, 44 with automatic resume support
!python run_all.py --stage C --seeds 42 43 44 --data_root /content/tn3k --checkpoints_root $CKPT_ROOT

## 📊 DELIVERABLES INSPECTION & DISPLAY

### Deliverable 1: 12-Panel Comprehensive Training Curves

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

curve_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/training_curves_E1_E4.png'
if os.path.exists(curve_path):
    plt.figure(figsize=(20, 12))
    plt.imshow(Image.open(curve_path))
    plt.axis('off')
    plt.show()

### Deliverable 2: Final Test Comparison Table (E1 vs E2 vs E3 vs E4)

In [ ]:
import pandas as pd
csv_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/final_test_comparison.csv'
if os.path.exists(csv_path):
    display(pd.read_csv(csv_path))

### Deliverable 3: Lesion-Scale Stratification Analysis (Small, Medium, Large)

In [ ]:
csv_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/lesion_scale_analysis.csv'
if os.path.exists(csv_path):
    display(pd.read_csv(csv_path))

### Deliverable 4: Model Efficiency & Hardware Latency Benchmark

In [ ]:
csv_path = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/model_efficiency_benchmark.csv'
if os.path.exists(csv_path):
    display(pd.read_csv(csv_path))

### Deliverable 5: Qualitative Comparisons (Stratified 8 Cases, Best Gains, Failures)

In [ ]:
qual_8 = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/qualitative_comparison_8_cases.png'
qual_gains = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/qualitative_best_gains.png'
qual_fail = '/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables/qualitative_failures.png'

for title, p in [('Stratified 8 Representative Cases', qual_8),
                 ('Top 6 Greatest Improvement Cases (E4 vs E1)', qual_gains),
                 ('Challenging Failure Cases for E4', qual_fail)]:
    if os.path.exists(p):
        print(f"\n=== {title} ===")
        plt.figure(figsize=(18, 12))
        plt.imshow(Image.open(p))
        plt.axis('off')
        plt.show()

### Deliverable 6: Master Deliverables Asset Verification Checklist

In [ ]:
from pathlib import Path

root = Path('/content/drive/MyDrive/ML_UVnet_checkpoints/final_deliverables')
files_to_check = [
    'final_test_comparison.csv',
    'lesion_scale_analysis.csv',
    'model_efficiency_benchmark.csv',
    'training_summary.csv',
    'training_curves_E1_E4.png',
    'qualitative_comparison_8_cases.png',
    'qualitative_best_gains.png',
    'qualitative_failures.png',
    'qualitative_cases.csv',
]

print("====================================================")
print("           AI512 REPORT ASSET VERIFICATION          ")
print("====================================================")
all_passed = True
for name in files_to_check:
    p = root / name
    found = p.exists() and p.stat().st_size > 0
    status = "✅ FOUND" if found else "❌ MISSING"
    print(f"{status:10} | {name}")
    if not found:
        all_passed = False

print("----------------------------------------------------")
if all_passed:
    print("🎉 ALL DELIVERABLES SUCCESSFULLY GENERATED AND SAVED!")
else:
    print("⚠️ Some deliverables are missing. Please complete all experiments.")
print("====================================================")